# Train court pose model on PROJECTION-GENERATED labels (snapped-H pipeline)

**Goal:** benchmark the new label pipeline. Labels come from 1,858 human-verified, line-snapped
homographies (median line residual 0.52 px) — denser and cleaner than the original court_det2 boxes.
Two formats, same split, same recipe as the old `court_aug` run (heavy HSV aug), so results are
directly comparable to `models/court_kp33_aug.pt`:

| DATASET | scheme | kpt_shape | run name |
|---|---|---|---|
| `court_pose33` | 33 court vertices (benchmark vs old model) | [33,3] | `pose33_snapped` |
| `court_pose_grid` | 13×7 court grid, 91 pts | [91,3] | `grid_snapped` |

**Before running:** GPU runtime, and upload `court_pose33_colab.zip` (and/or
`court_pose_grid_colab.zip`, 352 MB each) to the TOP LEVEL of My Drive.

**Cells:** 1 GPU → 2 set `DATASET`, mount+unzip (expect 1579/279) → 3 train → 4 confirm weights → 5 resume after disconnect.

When done: download `weights/best.pt` → save into the project as `models/court_kp33_snapped.pt`
(33) or `models/court_grid_snapped.pt` (grid). Do NOT overwrite the old models — we benchmark first.

In [ ]:
# 1) Confirm GPU
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "— NO GPU, set Runtime → GPU")

In [ ]:
# 2) Pick the dataset and load it from Google Drive.
DATASET = "court_pose33"          # <-- or "court_pose_grid" for the 91-pt grid run

!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
ZIP = f"/content/drive/MyDrive/{DATASET}_colab.zip"
import os; assert os.path.exists(ZIP), f"Not found: {ZIP} -> upload the zip to My Drive root first"
!rm -rf /content/$DATASET && mkdir -p /content/$DATASET
!unzip -q "$ZIP" -d /content/$DATASET
ROOT = f"/content/{DATASET}"
print(open(f"{ROOT}/data_colab.yaml").read()[:250])
import glob
print("train imgs:", len(glob.glob(f"{ROOT}/images/train/*")),
      "| val:", len(glob.glob(f"{ROOT}/images/val/*")))     # expect 1579 / 279

In [ ]:
# 3) Train yolov8m-POSE — IDENTICAL recipe to the old court_aug run (heavy HSV appearance aug,
#    same geometry aug), so old-vs-new isolates the LABELS. Checkpoints -> Drive (disconnect-safe).
RUN = {"court_pose33": "pose33_snapped", "court_pose_grid": "grid_snapped"}.get(DATASET, DATASET + "_run")
from ultralytics import YOLO
YOLO("yolov8m-pose.pt").train(
    data=f"{ROOT}/data_colab.yaml", epochs=300, imgsz=640, batch=16, device=0, seed=42,
    project="/content/drive/MyDrive/court_kp_runs", name=RUN, patience=30,
    fliplr=0.5, degrees=0.0, translate=0.05, scale=0.2, mosaic=0.0,   # geometry (unchanged)
    hsv_h=0.03, hsv_s=0.9, hsv_v=0.7)                                 # heavy color/brightness aug
print(f"\nDONE -> /content/drive/MyDrive/court_kp_runs/{RUN}/weights/best.pt")

In [ ]:
# 4) Weights are ALREADY on Drive (project pointed there). Confirm + where to put them.
import glob
RUN = {"court_pose33": "pose33_snapped", "court_pose_grid": "grid_snapped"}.get(DATASET, DATASET + "_run")
print("best.pt:", glob.glob(f"/content/drive/MyDrive/court_kp_runs/{RUN}*/weights/best.pt"))
print("last.pt:", glob.glob(f"/content/drive/MyDrive/court_kp_runs/{RUN}*/weights/last.pt"))
print("\n-> download best.pt; save as models/court_kp33_snapped.pt (33) or models/court_grid_snapped.pt (grid)")

In [ ]:
# 5) ===== RESUME AFTER A DISCONNECT =====
# Reconnect to a GPU runtime, set DATASET below, then run ONLY this cell.
DATASET = "court_pose33"          # <-- must match the interrupted run
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists(f"/content/{DATASET}/data_colab.yaml"):
    !rm -rf /content/$DATASET && mkdir -p /content/$DATASET
    !unzip -q "/content/drive/MyDrive/{DATASET}_colab.zip" -d /content/$DATASET
RUN = {"court_pose33": "pose33_snapped", "court_pose_grid": "grid_snapped"}.get(DATASET, DATASET + "_run")
LAST = f"/content/drive/MyDrive/court_kp_runs/{RUN}/weights/last.pt"
assert os.path.exists(LAST), f"No checkpoint at {LAST}"
from ultralytics import YOLO
YOLO(LAST).train(resume=True)